# Step 5 · Final figures and key numbers

Turns the result tables of Steps 3 and 4 into three figures ready for the brief, and collects every number the brief uses into one table that names the file it comes from.

This notebook trains no model and needs no Earth Engine; it runs in under a minute.

| Figure | Content |
| --- | --- |
| Figure 1 | Map of one held-out hurricane (Helene 2024 by default): observed outage duration against the model's prediction |
| Figure 2 | (a) skill of each feature set on a hurricane it has not seen; (b) counties with much and little forest under the same gusts |
| Figure 3 | Night lights: (a) against the share of customers with power in EAGLE-I; (b) against the model's prediction |

The `us_flood_outage` folder needs all the `step4_*.csv` files of Step 4, plus `step3a_features_county_event.csv`, `step3b_features_county_event.csv` and `step3b_ntl_by_power_bin.csv`. If `step3a_cache/tracks.csv` is there, the map shows the storm track; if not, it is left out. `step3b_flood_by_event.csv` is likewise used if present.

County boundaries are downloaded once from a public file on GitHub (derived from the U.S. Census Bureau's cartographic boundaries) and kept in `step5_cache`.

In [ ]:
# 1. Environment, output folder, chart style
import os, sys, urllib.request
import numpy as np
import pandas as pd
import scipy, matplotlib
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from scipy.stats import spearmanr
from shapely.geometry import box
from IPython.display import display
import geopandas as gpd

try:                                    # in Colab: mount Google Drive
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/us_flood_outage"
except ImportError:                     # in local Jupyter: use the current folder
    OUT = "us_flood_outage" if os.path.isdir("us_flood_outage") else "../us_flood_outage"    # next to the notebook, or one level up
FIG, CACHE = f"{OUT}/step5_figures", f"{OUT}/step5_cache"
os.makedirs(FIG, exist_ok=True); os.makedirs(CACHE, exist_ok=True)
print("Figures go to:", FIG)
print("python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__, "| scipy", scipy.__version__,
      "| matplotlib", matplotlib.__version__, "| geopandas", gpd.__version__)

C1, C2 = "#2a78d6", "#eb6834"                                              # the two series colors: blue, orange
SEQ = ["#cde2fb", "#9ec5f4", "#6da7ec", "#2a78d6", "#1c5cab", "#0d366b"]   # one hue from light to dark, for the duration classes on the map
NEUTRAL = "#f0efec"                                                        # counties outside the sample
INK, MUTED, GRID, SURFACE, FAINT = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb", "#b9b8b2"
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 8, "axes.titlesize": 9, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.labelsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 7.5, "legend.frameon": False,
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "text.color": INK, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "figure.dpi": 130, "savefig.dpi": 300, "savefig.bbox": "tight", "pdf.fonttype": 42,
})

def save(fig, name):
    # every figure is saved twice: PNG (300 dpi) and PDF (vector)
    for ext in ("png", "pdf"):
        fig.savefig(f"{FIG}/{name}.{ext}")
    plt.show()

## Parameters

In [ ]:
# 2. Parameters
MAP_EVENT = "helene2024"          # the hurricane shown on the map
REFERENCE = "wind_rain_place"     # the reference model (same as in Step 4)
LONG_H    = 24                    # threshold for a "long outage" (hours)
TOP_SHARE = 0.20                  # share of counties counted as "longest predicted"
MIN_GAIN, MIN_EVENTS = 0.02, 6    # the rule of Step 4: mean Spearman up by at least 0.02, and better in at least 6 of 9
HYP_LABELS = {"H1": "rain, added to wind", "H2": "place, added to wind and rain", "H3": "flood, added to place", "H4": "embeddings, added to everything",
              "H5": "embeddings instead of place"}
BINS       = [-np.inf, 1, 6, 24, 72, 168, np.inf]     # duration classes on the map (hours), closed on the left
BIN_LABELS = ["under 1 h", "1–6 h", "6–24 h", "1–3 days", "3–7 days", "over 7 days"]
MAP_PAD_KM = 60                   # margin around the mapped counties (km)
GEO_URL = "https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json"

EVENT_NAMES = {"irma2017": "Irma 2017", "florence2018": "Florence 2018", "michael2018": "Michael 2018", "laura2020": "Laura 2020",
               "zeta2020": "Zeta 2020", "ida2021": "Ida 2021", "ian2022": "Ian 2022", "beryl2024": "Beryl 2024", "helene2024": "Helene 2024"}
SET_LABELS = {"wind": "Wind", "wind_rain": "+ rain", "wind_rain_place": "+ place (land, people)", "wind_rain_place_flood": "+ flood",
              "wind_rain_place_flood_emb": "+ satellite embeddings", "wind_rain_emb": "Wind + rain + embeddings"}
STATE_ABBR = {"01": "AL", "04": "AZ", "05": "AR", "06": "CA", "08": "CO", "09": "CT", "10": "DE", "11": "DC", "12": "FL", "13": "GA", "16": "ID",
              "17": "IL", "18": "IN", "19": "IA", "20": "KS", "21": "KY", "22": "LA", "23": "ME", "24": "MD", "25": "MA", "26": "MI", "27": "MN",
              "28": "MS", "29": "MO", "30": "MT", "31": "NE", "32": "NV", "33": "NH", "34": "NJ", "35": "NM", "36": "NY", "37": "NC", "38": "ND",
              "39": "OH", "40": "OK", "41": "OR", "42": "PA", "44": "RI", "45": "SC", "46": "SD", "47": "TN", "48": "TX", "49": "UT", "50": "VT",
              "51": "VA", "53": "WA", "54": "WV", "55": "WI", "56": "WY"}

## A. Load the result tables

Everything comes from files that Steps 3 and 4 have already written; no model is recomputed here.

In [ ]:
# 3. Load the result tables
P   = pd.read_csv(f"{OUT}/step4_predictions.csv", dtype={"fips": str})            # held-out predictions for every county-event
S   = pd.read_csv(f"{OUT}/step4_metrics_summary.csv", index_col=0)
SE  = pd.read_csv(f"{OUT}/step4_spearman_by_event.csv", index_col=0)
CMP = pd.read_csv(f"{OUT}/step4_feature_set_comparison.csv", index_col=0)
ST  = pd.read_csv(f"{OUT}/step4_strict_evaluation.csv", index_col=0)
RB  = pd.read_csv(f"{OUT}/step4_robustness.csv", index_col=0)
IM  = pd.read_csv(f"{OUT}/step4_importance.csv", index_col=0)
GF  = pd.read_csv(f"{OUT}/step4_gust_forest.csv")
NT  = pd.read_csv(f"{OUT}/step4_ntl_check.csv", index_col=0)
NB  = pd.read_csv(f"{OUT}/step3b_ntl_by_power_bin.csv", index_col=0)
F3A = pd.read_csv(f"{OUT}/step3a_features_county_event.csv", dtype={"fips": str})
F3B = pd.read_csv(f"{OUT}/step3b_features_county_event.csv", dtype={"fips": str})

P["y"]          = np.log1p(P["hours_above"])
P["pred"]       = P[f"pred_{REFERENCE}"]                       # held-out prediction of the reference model, log(1 + hours)
P["pred_hours"] = np.expm1(P["pred"]).clip(lower=0)
P = P.merge(F3A[["event", "fips", "forest_pct", "gust_mean_ms"]], on=["event", "fips"], how="left", validate="1:1")
P = P.merge(F3B[["event", "fips", "ntl_nights_7d", "ntl_min_pct_7d", "s1_timely"]], on=["event", "fips"], how="left", validate="1:1")
EVENTS = [k for k in EVENT_NAMES if (P["event"] == k).any()]
SETS   = list(S.index)
assert set(SETS) == set(SET_LABELS) and P["forest_pct"].notna().all()
print(f"{len(P):,} county-events, {P['fips'].nunique()} counties, {len(EVENTS)} hurricanes")

## B. Figure 1: map of one held-out hurricane

On the left, the observed outage duration recorded by EAGLE-I; on the right, the prediction of the reference model. That model is trained on the other 8 hurricanes and has not seen the one on the map. Both sides use the same classes and colors; darker means a longer outage. Gray counties are outside the sample: either outside the states chosen for this hurricane, or not inside the wind swath and with less than 100 mm of rain in 3 days.

The coordinate system is the equal-area conic projection commonly used for the contiguous United States (EPSG:5070), which keeps county areas undistorted.

In [ ]:
# 4. County boundaries: downloaded once, kept locally
geo_path = f"{CACHE}/counties.geojson"
if not os.path.exists(geo_path):
    urllib.request.urlretrieve(GEO_URL, geo_path)
counties = gpd.read_file(geo_path)
counties["fips"] = counties["STATE"] + counties["COUNTY"]
counties = counties[counties["STATE"].isin(list(STATE_ABBR))][["fips", "STATE", "NAME", "geometry"]].to_crs(5070)
states = counties.dissolve(by="STATE")
missing = sorted(set(P["fips"]) - set(counties["fips"]))
print(f"{len(counties):,} county outlines; sample counties without an outline: {len(missing)}")

track_path = f"{OUT}/step3a_cache/tracks.csv"
track = None
if os.path.exists(track_path):
    t = pd.read_csv(track_path)
    t = t[t["event"] == MAP_EVENT].sort_values("time")
    if len(t):
        track = gpd.GeoSeries(gpd.points_from_xy(t["lon"], t["lat"]), crs=4326).to_crs(5070)
print("storm track found" if track is not None else "no storm track file, the map is drawn without it")

In [ ]:
# 5. Figure 1: observed outage duration against the held-out prediction
m = P[P["event"] == MAP_EVENT].copy()
m["obs_class"]  = pd.cut(m["hours_above"], BINS, labels=False, right=False)
m["pred_class"] = pd.cut(m["pred_hours"], BINS, labels=False, right=False)
shape  = counties.merge(m[["fips", "hours_above", "pred_hours", "obs_class", "pred_class"]], on="fips", how="left")
inside = shape["obs_class"].notna()
x0, y0, x1, y1 = shape[inside].total_bounds
pad = MAP_PAD_KM * 1000
frame = box(x0 - pad, y0 - pad, x1 + pad, y1 + pad)
rho_map = SE.loc[MAP_EVENT, REFERENCE]

same  = float((m["obs_class"] == m["pred_class"]).mean() * 100)
near  = float(((m["obs_class"] - m["pred_class"]).abs() <= 1).mean() * 100)
under = float((m["pred_class"] < m["obs_class"]).mean() * 100)

map_w = 2.4                                                   # width of each map panel (inches)
map_h = map_w * (y1 - y0 + 2 * pad) / (x1 - x0 + 2 * pad)
height = map_h + 0.8
fig = plt.figure(figsize=(7.0, height))
gs = fig.add_gridspec(1, 3, width_ratios=[map_w, map_w, 7.0 - 2 * map_w - 0.2], left=0.01, right=0.995, bottom=0.01, top=map_h / height, wspace=0.02)
for k, (col, title) in enumerate([("obs_class", "Observed (EAGLE-I)"), ("pred_class", "Predicted, hurricane held out")]):
    ax = fig.add_subplot(gs[0, k])
    shape[~inside].plot(ax=ax, color=NEUTRAL, edgecolor=SURFACE, linewidth=0.25)
    shape[inside].plot(ax=ax, color=[SEQ[int(c)] for c in shape.loc[inside, col]], edgecolor=SURFACE, linewidth=0.25)
    states.boundary.plot(ax=ax, color=MUTED, linewidth=0.5)
    if track is not None:
        ax.plot(track.x, track.y, color=INK, lw=1.1, ls=(0, (4, 2)))
    for code_, geom in states.geometry.intersection(frame).items():
        if geom.area > 0.025 * frame.area:
            pt = geom.representative_point()
            ax.text(pt.x, pt.y, STATE_ABBR[code_], ha="center", va="center", fontsize=6.5, color=INK,
                    path_effects=[pe.withStroke(linewidth=1.8, foreground=SURFACE)])
    ax.set_xlim(x0 - pad, x1 + pad); ax.set_ylim(y0 - pad, y1 + pad); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(title, fontsize=8.5, pad=3)
lax = fig.add_subplot(gs[0, 2]); lax.axis("off")
handles = [Patch(facecolor=c, edgecolor="none", label=l) for c, l in zip(SEQ, BIN_LABELS)] + [Patch(facecolor=NEUTRAL, edgecolor="none", label="not in the sample")]
if track is not None:
    handles.append(Line2D([], [], color=INK, lw=1.1, ls=(0, (4, 2)), label="storm track"))
lax.legend(handles=handles, loc="upper left", bbox_to_anchor=(0.0, 0.97), title="Hours with at least 10%\nof customers out", title_fontsize=7.5,
           alignment="left", handlelength=1.3, labelspacing=0.45, borderaxespad=0)
lax.text(0.0, 0.50, f"{int(inside.sum())} counties\n\nRank correlation between\nthe two maps: {rho_map:.2f}\n\nSame class: {same:.0f}%\nWithin one class: {near:.0f}%\n"
         f"Predicted lower: {under:.0f}%", transform=lax.transAxes, va="top", color=MUTED, fontsize=7.5, linespacing=1.25)
fig.suptitle(f"{EVENT_NAMES[MAP_EVENT]}: outage duration by county, observed and predicted", x=0.012, y=0.995, ha="left", va="top", fontweight="bold", fontsize=9.5)
fig.text(0.012, 1 - 0.29 / height, f"The prediction comes from a model trained on the other {len(EVENTS) - 1} hurricanes", color=MUTED, fontsize=7.5, va="top")
save(fig, "fig1_map")

print(f"{EVENT_NAMES[MAP_EVENT]}: counties where prediction and observation fall in the same class {same:.0f}%, within one class {near:.0f}%, predicted lower than observed {under:.0f}%")

## C. Figure 2: skill of each feature set; gust and forest

**(a)** Each row is one feature set. Gray dots are the Spearman of each of the 9 held-out hurricanes, the filled blue dot is their mean, and the open blue dot is the mean under the strict evaluation (training data also leave out every county of the hurricane being predicted). The vertical line is the baseline that trains no model and ranks counties by gust alone. The two columns on the right give the mean and the strict value.

**(b)** Counties are split into thirds by forest share and the lowest and highest thirds are kept, then binned by gust. Dots are the median observed outage duration in each bin, lines the median held-out prediction of the reference model. The panel pools the 9 hurricanes, and the forested counties come mostly from a few of them, so forest cannot be separated from the hurricane itself.

In [ ]:
# 6. Figure 2: (a) skill of each feature set on unseen hurricanes; (b) gust, forest and outage duration
base_gust = float(SE["gust_only"].mean())
lo_cut, hi_cut = P["forest_pct"].quantile([1 / 3, 2 / 3])

fig, (ax, bx) = plt.subplots(1, 2, figsize=(7.0, 3.0), gridspec_kw={"width_ratios": [1.0, 1.08], "wspace": 0.58, "left": 0.2, "right": 0.955,
                                                                    "top": 0.86, "bottom": 0.25})
rows = np.arange(len(SETS))[::-1]
ax.axvline(base_gust, color=FAINT, lw=1)
ax.text(base_gust, len(SETS) - 0.42, f"gust alone {base_gust:.2f}", ha="center", va="bottom", color=MUTED, fontsize=7)
for yi, s in zip(rows, SETS):
    v = SE[s].values
    ax.plot(v, yi + np.linspace(0.26, 0.12, len(v)), ls="none", marker="o", ms=3, mfc=FAINT, mec="none")
    ax.plot(ST.loc[s, "strict"], yi - 0.16, marker="o", ms=5.5, mfc=SURFACE, mec=C1, mew=1.3)
    ax.plot(S.loc[s, "mean_spearman"], yi - 0.16, marker="o", ms=6.5, mfc=C1, mec=SURFACE, mew=1)
    ax.text(1.04, yi - 0.16, f"{S.loc[s, 'mean_spearman']:.2f}", transform=ax.get_yaxis_transform(), ha="left", va="center", color=INK)
    ax.text(1.20, yi - 0.16, f"{ST.loc[s, 'strict']:.2f}", transform=ax.get_yaxis_transform(), ha="left", va="center", color=MUTED)
ax.text(1.04, len(SETS) - 0.42, "mean", transform=ax.get_yaxis_transform(), ha="left", va="bottom", color=MUTED, fontsize=7)
ax.text(1.20, len(SETS) - 0.42, "strict", transform=ax.get_yaxis_transform(), ha="left", va="bottom", color=MUTED, fontsize=7)
ax.axhline(0.5, color=GRID, lw=0.8)
ax.set_yticks(rows); ax.set_yticklabels([SET_LABELS[s] for s in SETS]); ax.tick_params(axis="y", length=0)
ax.set_ylim(-0.6, len(SETS) - 0.1); ax.grid(axis="y", visible=False)
ax.set_xlim(min(float(SE[SETS].min().min()), base_gust) - 0.03, max(float(SE[SETS].max().max()), 0.9) + 0.02)
ax.set_xlabel("Rank correlation, predicted vs observed duration")
ax.set_title("(a) Skill on a hurricane the model has not seen", pad=16, x=-0.69)
ax.legend(handles=[Line2D([], [], ls="none", marker="o", ms=3, mfc=FAINT, mec="none", label="each held-out hurricane"),
                   Line2D([], [], ls="none", marker="o", ms=6.5, mfc=C1, mec=SURFACE, mew=1, label="mean of 9"),
                   Line2D([], [], ls="none", marker="o", ms=5.5, mfc=SURFACE, mec=C1, mew=1.3, label="mean, strict")],
          loc="upper center", bbox_to_anchor=(0.3, -0.23), ncol=3, columnspacing=1.0, handletextpad=0.2)

for label, color in [("Little forest", C1), ("Much forest", C2)]:
    t = GF[GF["group"] == label]
    bx.plot(t["gust"], t["predicted_hours"] / 24, color=color, lw=1.6)
    bx.plot(t["gust"], t["actual_hours"] / 24, ls="none", marker="o", ms=5.5, mfc=color, mec=SURFACE, mew=1)
    bx.annotate(label, (t["gust"].iloc[-1], t["actual_hours"].iloc[-1] / 24), xytext=(6, 0), textcoords="offset points", va="center", color=INK, fontsize=7.5)
bx.set_xlim(right=bx.get_xlim()[1] + 6.5); bx.set_ylim(bottom=0)
bx.set_xlabel("Mean of maximum gust in the county (m/s)"); bx.set_ylabel("Median days with 10%+ out")
bx.set_title("(b) Outage duration by gust and forest", pad=16)
bx.text(0, 1.03, f"Forest on ≤ {lo_cut:.0f}% and ≥ {hi_cut:.0f}% of land; all hurricanes pooled", transform=bx.transAxes, color=MUTED, fontsize=7)
bx.legend(handles=[Line2D([], [], ls="none", marker="o", ms=5.5, mfc=MUTED, mec=SURFACE, mew=1, label="observed"),
                   Line2D([], [], color=MUTED, lw=1.6, label="held-out prediction")],
          loc="upper center", bbox_to_anchor=(0.5, -0.23), ncol=2, columnspacing=1.2, handlelength=1.4)
save(fig, "fig2_skill_and_forest")

## D. Figure 3: night lights

Night lights are not in the model; here they serve as an independent yardstick.

**(a)** Clear county-nights are grouped by the share of customers with power that night (EAGLE-I), to see what percentage of normal brightness the satellite recorded. Night lights dim clearly only when a large share of customers is out; with 50% to 80% of customers on power, the median falls only to just under nine tenths of normal.

**(b)** Counties are split into quarters by the reference model's held-out prediction, to see how far night lights fell within 7 days of landfall. The quarter with the longest predicted outages is the darkest: a median of 70% of normal, against 90% to 92% in the other three.

Dots are medians, bars the middle half.

In [ ]:
# 7. Figure 3: (a) night lights against the share with power; (b) night lights against the model's prediction
ntl = P[P["ntl_nights_7d"] >= 1].copy()
ntl["dimming"] = 100 - ntl["ntl_min_pct_7d"]
rho_ntl_pred = float(spearmanr(ntl["pred"], ntl["dimming"])[0])
rho_ntl_obs  = float(spearmanr(ntl["hours_above"], ntl["dimming"])[0])

def dot_and_bar(axis, tab, labels):
    x = np.arange(len(tab))
    axis.axhline(100, color=FAINT, lw=1)
    axis.vlines(x, tab["ntl_q25"], tab["ntl_q75"], color=C2, lw=4, alpha=0.35)
    axis.plot(x, tab["ntl_median"], ls="none", marker="o", ms=6.5, mfc=C2, mec=SURFACE, mew=1.2)
    for xi, med in enumerate(tab["ntl_median"]):
        axis.text(xi + 0.14, med, f"{med:.0f}%", va="center", color=INK, fontsize=7.5)
    axis.set_xticks(x); axis.set_xticklabels(labels); axis.set_xlim(-0.5, len(x) - 0.3); axis.set_ylim(0, 125); axis.grid(axis="x", visible=False)

fig, (ax, bx) = plt.subplots(1, 2, figsize=(7.0, 2.8), gridspec_kw={"width_ratios": [1.3, 1.0], "wspace": 0.36, "left": 0.075, "right": 0.985,
                                                                    "top": 0.84, "bottom": 0.2})
dot_and_bar(ax, NB, [str(k).replace("Not affected", "county not\naffected").replace("Under ", "under\n") for k in NB.index])
ax.set_xlabel("Customers with power that night (EAGLE-I)"); ax.set_ylabel("Night lights, percent of normal")
ax.set_title("(a) Night lights dim clearly only in large outages", pad=16)
ax.text(0, 1.03, f"{int(NB['nights'].sum()):,} clear county-nights after landfall", transform=ax.transAxes, color=MUTED, fontsize=7)
dot_and_bar(bx, NT, ["shortest", "second", "third", "longest"])
bx.set_xlabel("Quarter of county-hurricane pairs, by predicted duration"); bx.set_ylabel("Lowest night lights within 7 days,\npercent of normal")
bx.set_title("(b) Night lights against the prediction", pad=16)
bx.text(0, 1.03, f"{len(ntl):,} county-events; held-out predictions", transform=bx.transAxes, color=MUTED, fontsize=7)
save(fig, "fig3_night_lights")
print(f"Spearman of predicted duration and night-light dimming: {rho_ntl_pred:.3f}; of observed duration and dimming: {rho_ntl_obs:.3f}")

## E. Overall severity of each hurricane

The model ranks the counties of a new hurricane well, but its estimate of how severe that hurricane is overall is unstable. The table below lists, by hurricane, the observed and predicted median duration, the share of counties over 24 hours, and whether predictions are too high or too low on average (`bias_log`, on the log scale; negative means underestimated).

It also gives one number of practical relevance: among the 20% of counties with the longest predicted outages in each hurricane, the share that exceeded 24 hours.

This section describes the held-out predictions of Step 4; it is not one of the checks fixed in advance in Step 4.

In [ ]:
# 8. By hurricane: observed and predicted overall level
g = P.groupby("event", sort=False)
severity = pd.DataFrame({
    "counties": g.size(),
    "observed_median_h": g["hours_above"].median(),
    "predicted_median_h": g["pred_hours"].median(),
    "observed_over24_pct": g["hours_above"].apply(lambda s: float((s >= LONG_H).mean() * 100)),
    "predicted_over24_pct": g["pred_hours"].apply(lambda s: float((s >= LONG_H).mean() * 100)),
    "bias_log": g.apply(lambda d: float((d["pred"] - d["y"]).mean()), include_groups=False),
    "top_share_over24_pct": g.apply(lambda d: float((d.loc[d["pred"] >= d["pred"].quantile(1 - TOP_SHARE), "hours_above"] >= LONG_H).mean() * 100), include_groups=False),
}).loc[EVENTS]
severity.index = [EVENT_NAMES[e] for e in severity.index]
severity.to_csv(f"{OUT}/step5_severity_by_event.csv")
top_hit = float(severity["top_share_over24_pct"].mean())
print(f"Among the {TOP_SHARE:.0%} of counties with the longest predicted outages in each hurricane, the share that exceeded {LONG_H} hours, mean of 9: {top_hit:.0f}%")
severity.round(2)

## F. The five comparisons under the strict test and with ridge regression

The decision rule of Step 4 was written for the ordinary leave-one-hurricane-out evaluation. Here the same five comparisons are also computed under the strict evaluation (training data also leave out every county of the hurricane being predicted), and the two are listed side by side. A third set, `standard_ridge`, swaps the model for ridge regression under the ordinary evaluation, to see whether the conclusions depend on the tree model. If the folder holds `check_scikit-learn_1.9.1/step4_predictions.csv` (predictions from another scikit-learn version), the same comparisons are repeated on it and written to `step5_version_check.csv`.

`meets_rule` applies the rule of Step 4 unchanged: H1 to H4 need a gain of at least 0.02 and at least 6 of 9 hurricanes better; H5 needs to be no more than 0.02 lower.

If a feature group clears the bar under only one of the two evaluations, it cannot be called an "improvement"; its contribution is near the threshold. The strict evaluation also trains on less data, so the difference between the two cannot all be put down to "location memory".

The strict and ridge parts of this table were added after seeing the results of Step 4; they are not part of the decisions fixed in advance.

In [ ]:
# 9. The five comparisons: standard and strict evaluation
def spearman_by_event(col, table):
    return pd.Series({e: float(spearmanr(d[col], d["hours_above"])[0]) for e, d in table.groupby("event", sort=False)})

def comparisons(table):
    rows = []
    for hyp, r in CMP.iterrows():
        for evaluation, prefix in [("standard", "pred_"), ("strict", "strict_"), ("standard_ridge", "ridge_")]:
            a, b = spearman_by_event(prefix + r["from"], table), spearman_by_event(prefix + r["to"], table)
            gain, better = float((b - a).mean()), int((b > a).sum())
            rows.append({"hypothesis": hyp, "comparison": HYP_LABELS[hyp], "evaluation": evaluation, "spearman_from": a.mean(), "spearman_to": b.mean(),
                         "gain": gain, "hurricanes_better": better,
                         "meets_rule": bool(gain >= -MIN_GAIN) if hyp == "H5" else bool(gain >= MIN_GAIN and better >= MIN_EVENTS)})
    return pd.DataFrame(rows)

both = comparisons(P)
std = both[both["evaluation"] == "standard"].set_index("hypothesis")
assert np.allclose(std["gain"], CMP["gain"], atol=1e-9) and (std["hurricanes_better"] == CMP["events_better"]).all()    # agrees with the Step 4 table
both.to_csv(f"{OUT}/step5_comparisons.csv", index=False)

# training rows: the standard test leaves out the held-out hurricane, the strict test also every county it touches
train_rows = pd.DataFrame({e: {"standard": int((P["event"] != e).sum()),
                               "strict": int(((P["event"] != e) & ~P["fips"].isin(set(P.loc[P["event"] == e, "fips"]))).sum())} for e in EVENTS}).T

# if the folder holds Step 4 predictions from another scikit-learn version, repeat the comparisons and see how far they move
alt_path = f"{OUT}/check_scikit-learn_1.9.1/step4_predictions.csv"
versions = None
if os.path.exists(alt_path):
    other = comparisons(pd.read_csv(alt_path, dtype={"fips": str}))
    versions = both.merge(other, on=["hypothesis", "comparison", "evaluation"], suffixes=("", "_other"))
    versions["gain_difference"] = (versions["gain"] - versions["gain_other"]).abs()
    versions["count_difference"] = (versions["hurricanes_better"] - versions["hurricanes_better_other"]).abs()
    versions.to_csv(f"{OUT}/step5_version_check.csv", index=False)
    display(versions[["hypothesis", "evaluation", "gain", "gain_other", "hurricanes_better", "hurricanes_better_other", "meets_rule", "meets_rule_other"]].round(3))
both.round(3)

## G. Numbers used in the brief

Every number quoted in the brief and the README, in one table that names the file it comes from.

In [ ]:
# 10. Table of key numbers
rows = []
def num(key, value, source, note=""):
    rows.append({"key": key, "value": value, "source": source, "note": note})

h = P["hours_above"]
num("county_events", len(P), "step4_predictions.csv", "exposed county-events with EAGLE-I records")
num("counties", P["fips"].nunique(), "step4_predictions.csv")
num("hurricanes", len(EVENTS), "step4_predictions.csv")
num("states", P["state"].nunique(), "step4_predictions.csv")
num("over_24h", int((h >= 24).sum()), "step4_predictions.csv", "county-events with 24+ hours at 10%+ out")
num("over_72h", int((h >= 72).sum()), "step4_predictions.csv")
num("never_reached_10pct", int((h == 0).sum()), "step4_predictions.csv")
num("never_reached_10pct_share", round(float((h == 0).mean() * 100), 1), "step4_predictions.csv", "percent of county-events")
num("gust_alone_mean_spearman", round(float(SE["gust_only"].mean()), 3), "step4_spearman_by_event.csv", "no model, counties ranked by gust")
for s in SETS:
    num(f"mean_spearman.{s}", round(float(S.loc[s, "mean_spearman"]), 3), "step4_metrics_summary.csv", "leave-one-hurricane-out, mean of 9")
    num(f"min_max_spearman.{s}", f"{SE[s].min():.2f} to {SE[s].max():.2f}", "step4_spearman_by_event.csv", "range over the 9 held-out hurricanes")
    num(f"strict_mean_spearman.{s}", round(float(ST.loc[s, "strict"]), 3), "step4_strict_evaluation.csv", "training also leaves out the test hurricane's counties")
    num(f"ridge_mean_spearman.{s}", round(float(S.loc[s, "ridge_mean_spearman"]), 3), "step4_metrics_summary.csv", "linear baseline")
num("mean_auc_24.reference", round(float(S.loc[REFERENCE, "mean_auc_24"]), 3), "step4_metrics_summary.csv", "classifier for 24+ hours, mean of 9")
for s in ["wind", REFERENCE, SETS[4]]:
    num(f"pooled_r2_log.{s}", round(float(S.loc[s, "pooled_r2_log"]), 3), "step4_metrics_summary.csv", "all hurricanes pooled, log(1 + hours): depends on storm-level severity too")
for hyp, r in CMP.iterrows():
    num(f"{hyp}.gain", round(float(r["gain"]), 3), "step4_feature_set_comparison.csv", f"{r['from']} -> {r['to']}")
    num(f"{hyp}.hurricanes_better", f"{int(r['events_better'])} of {int(r['events'])}", "step4_feature_set_comparison.csv")
    num(f"{hyp}.gain_ridge", round(float(r["gain_ridge"]), 3), "step4_feature_set_comparison.csv", "the same comparison with the linear model")
    num(f"{hyp}.verdict", r["verdict"], "step4_feature_set_comparison.csv", "rule fixed in advance: gain of 0.02 in at least 6 of 9 (H5: no more than 0.02 lower)")
for _, r in both.iterrows():
    num(f"{r['hypothesis']}.{r['evaluation']}.gain", round(float(r["gain"]), 3), "step5_comparisons.csv", r["comparison"])
    num(f"{r['hypothesis']}.{r['evaluation']}.hurricanes_better", f"{int(r['hurricanes_better'])} of {len(EVENTS)}", "step5_comparisons.csv")
    num(f"{r['hypothesis']}.{r['evaluation']}.meets_rule", bool(r["meets_rule"]), "step5_comparisons.csv", "the Step 4 rule applied unchanged")
num("train_rows.standard.median", int(train_rows["standard"].median()), "step4_predictions.csv", "county-events the model trains on, median of 9")
num("train_rows.strict.median", int(train_rows["strict"].median()), "step4_predictions.csv")
num("train_rows.strict.min", f"{int(train_rows['strict'].min())} ({EVENT_NAMES[train_rows['strict'].idxmin()]})", "step4_predictions.csv")
if versions is not None:
    tree = versions[versions["evaluation"] != "standard_ridge"]
    num("versions.largest_gain_difference", round(float(tree["gain_difference"].max()), 4), "step5_version_check.csv", "tree models, standard and strict tests, two scikit-learn versions")
    num("versions.largest_count_difference", int(tree["count_difference"].max()), "step5_version_check.csv", "hurricanes that improve")
    num("versions.ridge_gain_difference", round(float(versions.loc[versions["evaluation"] == "standard_ridge", "gain_difference"].max()), 6), "step5_version_check.csv")
    for _, r in tree[tree["meets_rule"] != tree["meets_rule_other"]].iterrows():
        num(f"versions.{r['hypothesis']}.{r['evaluation']}.gain_in_each_version", f"{r['gain']:.3f} and {r['gain_other']:.3f}", "step5_version_check.csv",
            "the rule gives a different answer in the two versions")
for k in ["wind_rain_place_flood_emb", "wind_rain_emb"]:
    num(f"mean_spearman_raw_64_dimensions.{k}", round(float(ST.loc[k, "standard_raw_64"]), 3), "step4_strict_evaluation.csv", "embeddings used as 64 raw dimensions instead of 16 components")
check_path = f"{OUT}/step3b_checklist.csv"
if os.path.exists(check_path):                                 # the checks fixed before Step 3b that failed
    for k, r in pd.read_csv(check_path).iterrows():
        if str(r.iloc[-1]).strip() == "fail":
            num(f"step3b_check_failed.row_{k}", f"{r.iloc[2]} (threshold {r.iloc[3]})", "step3b_checklist.csv", str(r.iloc[1])[:110])
timely = P.merge(F3A[["event", "fips", "rain_max3d_mm"]], on=["event", "fips"]).merge(F3B[["event", "fips", "flood_dry_pct", "flood_nocrop_pct"]], on=["event", "fips"])
timely = timely[timely["s1_timely"].astype(bool) & timely["flood_dry_pct"].notna()]
num("flood.rho_flooded_share_rain", round(float(spearmanr(timely["flood_dry_pct"], timely["rain_max3d_mm"])[0]), 2), "step3a_ and step3b_features_county_event.csv",
    f"{len(timely)} county-events with a timely radar image")
num("flood.rho_flooded_share_rain_without_cropland", round(float(spearmanr(timely["flood_nocrop_pct"], timely["rain_max3d_mm"])[0]), 2), "step3a_ and step3b_features_county_event.csv")
num("reference_model_scored_on_pairs_that_reached_10pct", round(float(np.mean([spearmanr(d["pred"], d["hours_above"])[0] for _, d in P[P["hours_above"] > 0].groupby("event", sort=False)])), 3),
    "step4_predictions.csv", "the reference model, scored only on pairs that reached 10%; robustness.affected_only is a model retrained on those pairs")
for k in ["second_denominator", "affected_only", "ridge", "shuffled_outcome"]:
    num(f"robustness.{k}", round(float(RB.loc[k, "mean_spearman"]), 3), "step4_robustness.csv")
for k, r in IM.iterrows():
    num(f"importance.{k}", round(float(r["mean_drop"]), 3), "step4_importance.csv", "drop in Spearman when the group is shuffled, full model")
num("forest.low_third_max_pct", round(float(lo_cut), 1), "step3a_features_county_event.csv", "forest share at the lower tercile")
num("forest.high_third_min_pct", round(float(hi_cut), 1), "step3a_features_county_event.csv", "forest share at the upper tercile")
for _, r in GF.iterrows():
    num(f"forest.{r['group']}.gust {r['gust_bin']}.observed_median_h", round(float(r["actual_hours"]), 1), "step4_gust_forest.csv", f"n = {int(r['n'])}")
bin_ = P[(P["gust_mean_ms"] > 27.5) & (P["gust_mean_ms"] <= 30)]            # the gust bin used as the example in the brief: which hurricanes each group comes from
for label, d in [("Little forest", bin_[bin_["forest_pct"] <= lo_cut]), ("Much forest", bin_[bin_["forest_pct"] >= hi_cut])]:
    mix = d["event"].value_counts().reindex(EVENTS).dropna().astype(int).sort_values(ascending=False, kind="stable")
    num(f"forest.{label}.gust (27.5, 30.0].pairs_by_hurricane", ", ".join(f"{EVENT_NAMES[k]}: {v}" for k, v in mix.items()), "step4_predictions.csv + step3a_features_county_event.csv",
        f"{len(d)} county-events; the comparison pools hurricanes")
num("ntl.rho_prediction_dimming", round(rho_ntl_pred, 3), "step4_predictions.csv + step3b_features_county_event.csv", f"{len(ntl)} county-events with a clear night within 7 days")
num("ntl.rho_observed_dimming", round(rho_ntl_obs, 3), "step4_predictions.csv + step3b_features_county_event.csv")
rho_ntl_ev = pd.Series({EVENT_NAMES[e]: float(spearmanr(d["pred"], d["dimming"])[0]) for e, d in ntl.groupby("event", sort=False)})
num("ntl.rho_prediction_dimming.lowest_hurricane", f"{rho_ntl_ev.min():.2f} ({rho_ntl_ev.idxmin()})", "step4_predictions.csv + step3b_features_county_event.csv")
num("ntl.rho_prediction_dimming.highest_hurricane", f"{rho_ntl_ev.max():.2f} ({rho_ntl_ev.idxmax()})", "step4_predictions.csv + step3b_features_county_event.csv")
num("ntl.rho_prediction_dimming.mean_of_hurricanes", round(float(rho_ntl_ev.mean()), 3), "step4_predictions.csv + step3b_features_county_event.csv")
for k, r in NT.iterrows():
    num(f"ntl.lowest_pct_of_normal.predicted_quarter.{k}", round(float(r["ntl_median"]), 1), "step4_ntl_check.csv", "median")
for k, r in NB.iterrows():
    num(f"ntl.pct_of_normal.customers_with_power.{k}", round(float(r["ntl_median"]), 1), "step3b_ntl_by_power_bin.csv", f"median over {int(r['nights'])} clear county-nights")
num("flood.rows_with_timely_radar_pct", round(float(P["s1_timely"].astype(float).mean() * 100), 1), "step3b_features_county_event.csv", "county-events with a Sentinel-1 image soon enough after landfall")
flood_path = f"{OUT}/step3b_flood_by_event.csv"
if os.path.exists(flood_path):                                 # the flood summary table of Step 3b, read if present
    FL = pd.read_csv(flood_path)
    num("flood.median_crop_share_pct.range_over_hurricanes", f"{FL['median_crop_share'].min():.0f} to {FL['median_crop_share'].max():.0f}", "step3b_flood_by_event.csv",
        "share of the flagged area that is cropland, median county, lowest and highest hurricane")
num("map.event", EVENT_NAMES[MAP_EVENT], "parameter MAP_EVENT")
num("map.counties", int(inside.sum()), "step4_predictions.csv")
num("map.rank_correlation", round(float(rho_map), 3), "step4_spearman_by_event.csv")
num("map.same_class_pct", round(same, 1), "step4_predictions.csv", "predicted and observed in the same duration class")
num("map.within_one_class_pct", round(near, 1), "step4_predictions.csv")
num("map.predicted_lower_pct", round(under, 1), "step4_predictions.csv")
week = m[m["hours_above"] >= 168]                              # counties of the mapped hurricane that were out for more than 7 days
num("map.observed_over_7d", len(week), "step4_predictions.csv", "counties with 168+ hours observed")
num("map.predicted_over_7d", int((m["pred_hours"] >= 168).sum()), "step4_predictions.csv", "counties with 168+ hours predicted")
for st, d in week.groupby("state"):
    num(f"map.observed_over_7d.{st}.counties", len(d), "step4_predictions.csv")
    num(f"map.observed_over_7d.{st}.observed_median_h", round(float(d["hours_above"].median()), 1), "step4_predictions.csv")
    num(f"map.observed_over_7d.{st}.predicted_median_h", round(float(d["pred_hours"].median()), 1), "step4_predictions.csv")
for ev, r in severity.iterrows():
    num(f"severity.{ev}.observed_over24_pct", round(float(r["observed_over24_pct"]), 1), "step5_severity_by_event.csv")
    num(f"severity.{ev}.predicted_over24_pct", round(float(r["predicted_over24_pct"]), 1), "step5_severity_by_event.csv")
num("top_share_over24_pct", round(top_hit, 1), "step5_severity_by_event.csv", f"top {TOP_SHARE:.0%} by predicted duration in each hurricane, mean of 9")

key_numbers = pd.DataFrame(rows)
key_numbers.to_csv(f"{OUT}/step5_key_numbers.csv", index=False)
print(f"{len(key_numbers)} numbers written to step5_key_numbers.csv")
with pd.option_context("display.max_rows", 200, "display.max_colwidth", 90):
    display(key_numbers)

## Output files

- `step5_figures/fig1_map`, `fig2_skill_and_forest`, `fig3_night_lights`, each as `.png` (300 dpi) and `.pdf` (vector)
- `step5_key_numbers.csv`: the numbers used in the brief and where they come from
- `step5_severity_by_event.csv`: observed and predicted overall level of each hurricane
- `step5_comparisons.csv`: the five comparisons under the standard test, the strict test and with ridge regression

To map another hurricane, change `MAP_EVENT` in cell 2 and run again.